# Deterministic Policy Gradient: differentiate through the action

The objective is to improve a deterministic actor $a=\mu_\theta(s)$ by following
the action derivative of a critic:

$$\nabla_\theta J\approx\mathbb E_{s\sim\rho^\beta}
[\nabla_\theta\mu_\theta(s)\nabla_a Q_\phi(s,a)|_{a=\mu_\theta(s)}].$$

$s$ is an observation, $a$ an action, $\theta$ actor weights, $\phi$ critic
weights, $Q$ the expected discounted return, and $\rho^\beta$ the state
visitation distribution of an exploratory behavior policy $\beta$.
The off-policy expression is the usual actor-critic approximation; exact
critic gradients and compatible approximation are separate theoretical issues.

This lesson isolates the chain rule on a one-step continuous-control task.
Use a linear actor with tanh and a quadratic critic, updating online without
replay or target networks. This is an educational DPG demonstration, not a
reproduction of every algorithm or compatibility condition in Silver et al.
DDPG, the next lesson, introduces deep networks, replay, target networks, and
long-horizon Pendulum control. Prerequisites: continuous Actor-Critic and TD
bootstrapping from the previous chapters.

## 1. Define a small continuous-control environment

At each episode draw $s\sim U(-1,1)$, choose $a\in[-1,1]$, and receive
$r=-(a-0.6s)^2$. The episode terminates immediately, so
$Q^*(s,a)=r$ and the optimal action is $a^*(s)=0.6s$.
This analytic reference lets us check the learned action gradient directly.
The constant $0.6$ is `TARGET_GAIN`; no environment derivatives are used to
train the actor or critic. Rendering compares target and selected action.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

TOTAL_TIMESTEPS = 5_000
ACTOR_LR = 2e-3
CRITIC_LR = 1e-2
ACTION_NOISE = 0.3
TARGET_GAIN = 0.6
GAMMA = 0.99
MAX_GRAD_NORM = 10.0
SEED = 7
EVALUATION_EPISODES = 5
RENDER_MODE = "human"
torch.set_num_threads(1)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


class TrackingEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"]}
    observation_space = gym.spaces.Box(-1.0, 1.0, (1,), dtype=np.float32)
    action_space = gym.spaces.Box(-1.0, 1.0, (1,), dtype=np.float32)

    def __init__(self, render_mode=None):
        self.render_mode = render_mode
        self.state = 0.0
        self.action = 0.0
        self.figure = None

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.state = float(self.np_random.uniform(-1, 1))
        self.action = 0.0
        return np.array([self.state], dtype=np.float32), {}

    def step(self, action):
        self.action = float(np.asarray(action).item())
        reward = -((self.action - TARGET_GAIN * self.state) ** 2)
        if self.render_mode == "human":
            self.render()
        return np.array([self.state], dtype=np.float32), reward, True, False, {}

    def render(self):
        if self.figure is None:
            self.figure, self.axes = plt.subplots(figsize=(7, 2))
        self.axes.clear()
        self.axes.scatter([TARGET_GAIN * self.state], [0], label="Target", s=100)
        self.axes.scatter([self.action], [0], label="Action", marker="x", s=100)
        self.axes.set(xlim=(-1, 1), ylim=(-0.5, 0.5), xlabel="Control action")
        self.axes.legend()
        self.figure.canvas.draw()
        if self.render_mode == "human":
            plt.show(block=False)
            plt.pause(0.2)
        else:
            return np.asarray(self.figure.canvas.buffer_rgba())[..., :3].copy()

    def close(self):
        if self.figure is not None:
            plt.close(self.figure)

## 2. Parameterize the actor and an action-differentiable critic

$$\mu_\theta(s)=\tanh(w s+b),\qquad
Q_\phi(s,a)=\phi^T[1,s,a,s^2,sa,a^2].$$

$\theta=(w,b)$ and the six coefficients $\phi$ are trainable. The quadratic
features can exactly represent this task's reward, making critic error easy
to diagnose. `Critic.forward` constructs those features from tensors; it
must retain the computation graph through $a$ during policy optimization.
Behavior adds independent Gaussian action noise to explore around the actor.

In [ ]:
actor = nn.Sequential(nn.Linear(1, 1), nn.Tanh())


class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.coefficients = nn.Parameter(torch.zeros(6))

    def forward(self, states, actions):
        features = torch.cat(
            (
                torch.ones_like(states),
                states,
                actions,
                states.square(),
                states * actions,
                actions.square(),
            ),
            -1,
        )
        return (features * self.coefficients).sum(-1, keepdim=True)


critic = Critic()
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=ACTOR_LR)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=CRITIC_LR)


@torch.no_grad()
def select_action(observation, deterministic=False):
    state = torch.tensor(observation, dtype=torch.float32).reshape(1, 1)
    action = actor(state).item()
    if not deterministic:
        action += rng.normal(0, ACTION_NOISE)
    return np.array([np.clip(action, -1, 1)], dtype=np.float32)

## 3. Fit the critic using a TD target

$$y=r+\gamma(1-d)Q_\phi(s',\mu_\theta(s')),\qquad
L_Q=(Q_\phi(s,a)-y)^2.$$

$d$ is one only for `terminated`; $s'$ is the next observation and $\gamma$
the discount. In this task every transition terminates, so $y=r$ exactly.
The general bootstrap is shown to connect this lesson with later control
algorithms. For a time-limit truncation it would remain active. Detach $y$
and fit the critic at the action actually taken by the noisy behavior policy.

In [ ]:
def update_critic(observation, action, reward, next_observation, terminated):
    state = torch.tensor(observation, dtype=torch.float32).reshape(1, 1)
    taken_action = torch.tensor(action, dtype=torch.float32).reshape(1, 1)
    next_state = torch.tensor(next_observation, dtype=torch.float32).reshape(1, 1)
    with torch.no_grad():
        target = reward + GAMMA * (1 - float(terminated)) * critic(
            next_state, actor(next_state)
        )
    loss = nn.functional.mse_loss(critic(state, taken_action), target)
    critic_optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(critic.parameters(), MAX_GRAD_NORM)
    critic_optimizer.step()
    critic_optimizer.zero_grad(set_to_none=True)
    return loss.item()

## 4. Apply the deterministic policy gradient

$$L_\mu=-Q_\phi(s,\mu_\theta(s)),\qquad
\nabla_\theta L_\mu=-\nabla_aQ_\phi(s,a)\nabla_\theta\mu_\theta(s).$$

Freeze the critic coefficients but keep its operations differentiable with
respect to the actor's action. `loss.backward()` computes the chain rule;
there is no log probability or likelihood-ratio estimator. The replay-free
state sample is still a behavior-state sample. Our one-step task has an
exogenous state distribution, so it avoids state-distribution complications.

In [ ]:
def update_actor(observation):
    state = torch.tensor(observation, dtype=torch.float32).reshape(1, 1)
    critic.requires_grad_(False)
    try:
        loss = -critic(state, actor(state)).mean()
        actor_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(actor.parameters(), MAX_GRAD_NORM)
        actor_optimizer.step()
    finally:
        critic.requires_grad_(True)
    return loss.item()

## 5. Explore and learn online

The behavior action is $a=\operatorname{clip}(\mu_\theta(s)+\epsilon,-1,1)$,
with $\epsilon\sim\mathcal N(0,\sigma^2)$ and $\sigma$=`ACTION_NOISE`.
Each transition produces one critic and actor update, then an environment
reset if either termination or truncation occurs. No experience is replayed.
The returned episode reward is the training signal; it includes exploration
noise, so deterministic evaluation can achieve higher returns.

In [ ]:
def train(total_timesteps):
    env = TrackingEnv()
    returns, critic_losses, actor_losses = [], [], []
    episode_return = 0.0
    try:
        observation, _ = env.reset(seed=SEED)
        for _ in range(total_timesteps):
            action = select_action(observation)
            next_observation, reward, terminated, truncated, _ = env.step(action)
            critic_losses.append(
                update_critic(observation, action, reward, next_observation, terminated)
            )
            actor_losses.append(update_actor(observation))
            episode_return += float(reward)
            observation = next_observation
            if terminated or truncated:
                returns.append(episode_return)
                episode_return = 0.0
                observation, _ = env.reset()
    finally:
        env.close()
    return returns, critic_losses, actor_losses


returns, critic_losses, actor_losses = train(TOTAL_TIMESTEPS)

## 6. Compare learning with the analytic solution

The moving mean uses $\bar R_k=\frac{1}{W}\sum_{j=k-W+1}^k R_j$ for window
size $W$. Compare $\mu_\theta(s)$ with $0.6s$ over a state grid.
For a fixed $s$, the true action derivative is
$\partial Q/\partial a=-2(a-0.6s)$; autograd should recover a similar curve
from the learned critic. A tanh-linear actor only approximates the ideal
linear action map, while the quadratic critic can represent Q exactly.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
window = min(100, len(returns))
axes[0].plot(returns, alpha=0.2)
if window:
    average = np.convolve(returns, np.ones(window) / window, mode="valid")
    axes[0].plot(np.arange(window - 1, len(returns)), average)
axes[0].set(xlabel="Episode", ylabel="Return", title="DPG training")
axes[1].plot(critic_losses)
axes[1].set(xlabel="Update", ylabel="MSE", title="Critic loss")
states = torch.linspace(-1, 1, 100).reshape(-1, 1)
with torch.no_grad():
    learned_actions = actor(states).numpy()
axes[2].plot(states.numpy(), learned_actions, label="Actor")
axes[2].plot(states.numpy(), TARGET_GAIN * states.numpy(), label="Optimal")
axes[2].set(xlabel="State", ylabel="Action", title="Learned control")
axes[2].legend()
plt.tight_layout()
plt.show()

fixed_state = torch.full((100, 1), 0.5)
actions = torch.linspace(-1, 1, 100).reshape(-1, 1).requires_grad_(True)
gradient = torch.autograd.grad(critic(fixed_state, actions).sum(), actions)[0]
plt.plot(actions.detach().numpy(), gradient.detach().numpy(), label="Learned dQ/da")
plt.plot(
    actions.detach().numpy(),
    -2 * (actions.detach().numpy() - TARGET_GAIN * 0.5),
    linestyle="--",
    label="Analytic dQ/da",
)
plt.xlabel("Action at state 0.5")
plt.ylabel("Action derivative")
plt.legend()
plt.show()

## 7. Evaluate in a separate rendered environment

Remove exploration noise and draw new states. A perfect controller obtains
zero reward on every episode; report how close the learned policy gets.
The rendered view compares the target and selected action. Set `RENDER_MODE`
to `"rgb_array"` for headless execution; the same renderer then returns pixels.

In [ ]:
evaluation_env = TrackingEnv(render_mode=RENDER_MODE)
evaluation_returns = []
try:
    for episode in range(EVALUATION_EPISODES):
        observation, _ = evaluation_env.reset(seed=1_000 + episode)
        action = select_action(observation, deterministic=True)
        _, reward, _, _, _ = evaluation_env.step(action)
        if RENDER_MODE == "rgb_array":
            frame = evaluation_env.render()
        evaluation_returns.append(float(reward))
finally:
    evaluation_env.close()
print("Evaluation returns:", evaluation_returns)
print(f"Mean deterministic return: {np.mean(evaluation_returns):.5f}")

## Next: DDPG

The deterministic gradient needs an accurate action derivative from Q.
Online neural bootstrapping on a long-horizon task is harder than this
quadratic one-step example. The next lesson introduces replay and target
networks to stabilize that setting.

[DPG paper](https://proceedings.mlr.press/v32/silver14.html) ·
[DDPG lesson](02_ddpg.ipynb) · [DPG guide](../../docs/algorithms/dpg.md)